In [1]:
import flax.linen as nn
import numpy as np
import optax
import jax.numpy as jnp
import swarmrl as srl
from swarmrl.models.interaction_model import Action
from swarmrl.observables import Observable
import jax
import h5py as hf
import matplotlib.pyplot as plt

import swarmrl.engine.espresso as espresso
from swarmrl.utils import utils

import pint

Could not find espressomd. Features will not be available


## System Definition

In [ ]:
def build_ising_lattice(n_rows: int, n_agents: int, buffer: int = 20):
    """
    Build an espresso simulation.
    """
    # Global properties
    box_length = (100 + 2 * buffer) * n_rows
    
    simulation_name = "ising-rl"
    seed = np.random.randint(0, 3453276453)
    
    
    # Create standard simulation.
    temperature = 10.
    n_colloids = n_agents

    ureg = pint.UnitRegistry()
    md_params = espresso.MDParams(
        ureg=ureg,
        fluid_dyn_viscosity=ureg.Quantity(8.9e-4, "pascal * second"),
        WCA_epsilon=ureg.Quantity(273.15, "kelvin") * ureg.boltzmann_constant,
        temperature=ureg.Quantity(temperature, "kelvin"),
        box_length=ureg.Quantity(box_length, "micrometer"),
        time_slice=ureg.Quantity(1.0, "second"),  # model timestep
        time_step=ureg.Quantity(0.01, "second"),  # integrator timestep
        write_interval=ureg.Quantity(2, "second"),
    )

    system_runner = srl.espresso.EspressoMD(
        md_params=md_params,
        n_dims=2,
        seed=seed,
        out_folder='./training',
        write_chunk_size=100,
    )

    system_runner.add_colloids(
        n_colloids,
        ureg.Quantity(2.14, "micrometer"),
        ureg.Quantity(np.array([500, 500, 0]), "micrometer"),
        ureg.Quantity(100, "micrometer"),
        type_colloid=0,
    )
    
    # Create rods
    center_locations = np.arange(buffer + 50, box_length - (50 + buffer), (100 + 2 * buffer))
    
    for location in center_locations:
        system_runner.add_rod(
            rod_center=ureg.Quantity([location, location, 0], "micrometer"),
            rod_length=ureg.Quantity(100, "micrometer"),
            rod_thickness=ureg.Quantity(100 / 59, "micrometer"),
            rod_start_angle=90.0,
            n_particles=59,
            friction_trans=ureg.Quantity(4.388999e-7, "newton * second / meter"),
            friction_rot=ureg.Quantity(6.902407326e-16, "newton * second * meter"),
            rod_particle_type=1,
        )


In [ ]:
n_slices = int(ureg.Quantity(3, "minute") / md_params.time_slice)
n_episodes = 300
episode_length = 50 #int(np.ceil(n_slices / 10))

## RL Problems

In [2]:
# Exploration policy
exploration_policy = srl.exploration_policies.RandomExploration(probability=0.2)

# Sampling strategy
sampling_strategy = srl.sampling_strategies.GumbelDistribution()

# Value function
value_function = srl.value_functions.ExpectedReturns(
    gamma=0.99, standardize=True
)

In [3]:
class IsingTask(srl.tasks.Task):
    """
    Task for minima detection.
    """
    def __init__(
        self,
        potential_fn: callable,
        particle_type: int = 0,
    ):
        """
        Constructor for the find origin task.

        Parameters
        ----------
        source : np.ndarray (default = (0, 0 0))
                Source of the gradient.
        potenial_fn : callable
                Function we are moving along.
        particle_type : int (default=0)

        """
        super().__init__(particle_type=particle_type)
        self.potential_fn = potential_fn

        # Class only attributes
        self._historic_potential = {}
        
    def initialize(self, colloids: List):
        """
        Prepare the task for running.

        Parameters
        ----------
        colloids : List[Colloid]
                List of colloids to be used in the task.

        Returns
        -------
        observable :
                Returns the observable required for the task.
        """
        for item in colloids:
            if item.type == self.particle_type:
                index = item.id
                function_value = self.potential_fn(item.pos)
                self._historic_potential[str(index)] = function_value
                
    def compute_colloid_reward(self, index: int, colloids):
        """
        Compute the reward for a single colloid.

        Parameters
        ----------
        index : int
                Index of the colloid to compute the reward for.
        colloids : list
                All colloids

        Returns
        -------
        reward : float
                Reward for the colloid.
        """
        colloid_id = colloids[index].id
        # Get the current position of the colloid
        current_position =colloids[index].pos
        current_function_value = self.potential_fn(current_position)
        past_function_value = self._historic_potential[str(colloid_id)]

        # Compute difference in scaled_distances
        delta = current_function_value - past_function_value

        # Compute the reward
        reward = -100 * delta + abs(current_function_value)

        # Update the historic position
        self._historic_potential[str(colloid_id)] = current_function_value

        return reward
    
    def __call__(self, colloids: List):
        """
        Compute the reward.

        In this case of this task, the observable itself is the gradient of the field
        that the colloid is swimming in. Therefore, the change is simply scaled and
        returned.

        Parameters
        ----------
        colloids : List[Colloid] (n_colloids, )
                List of colloids to be used in the task.

        Returns
        -------
        rewards : List[float] (n_colloids, )
                Rewards for each colloid.
        """
        colloid_indices = self.get_colloid_indices(colloids)

        return np.array(
            [self.compute_colloid_reward(index, colloids) for index in colloid_indices]
        )


NameError: name 'List' is not defined

In [4]:
class ColloidEmbedding(nn.Module):
    @nn.compact
    def __call__(self, x):
        return nn.Dense(features=2)(x)
    
class RodEmbedding(nn.Module):
    @nn.compact
    def __call__(self, x):
        return nn.Dense(features=2)(x)
    
class ActorNet(nn.Module):
    """A simple dense model."""
    
    def setup(self):
        self.colloid_embedding = ColloidEmbedding()
        self.rod_embeding = RodEmbedding()
    
    @nn.compact
    def __call__(self, x):
        colloid_embedding = self.colloid_embedding(x[:, 0])
        rod_embedding = self.rod_embeding(x[:, 1])
        
        x = colloid_embedding + rod_embedding
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=4)(x)
        return x
    
class CriticNet(nn.Module):
    """A simple dense model."""
    
    def setup(self):
        self.colloid_embedding = ColloidEmbedding()
        self.rod_embeding = RodEmbedding()
    
    @nn.compact
    def __call__(self, x):
        colloid_embedding = self.colloid_embedding(x[:, 0])
        rod_embedding = self.rod_embeding(x[:, 1])
        
        x = colloid_embedding + rod_embedding
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=1)(x)
        return x
    

In [5]:
actor = srl.networks.FlaxModel(
    flax_model=ActorNet(),
    optimizer=optax.adam(learning_rate=0.001),
    input_shape=(1, 5, 2),
    sampling_strategy=sampling_strategy,
    exploration_policy=exploration_policy,
)

critic = srl.networks.FlaxModel(
    flax_model=CriticNet(),
    optimizer=optax.adam(learning_rate=0.001),
    input_shape=(1, 5, 2),
)

translate = Action(force=10.0)
rotate_clockwise = Action(torque=np.array([0.0, 0.0, 10.0]))
rotate_counter_clockwise = Action(torque=np.array([0.0, 0.0, -10.0]))
do_nothing = Action()

actions = {
    "RotateClockwise": rotate_clockwise,
    "Translate": translate,
    "RotateCounterClockwise": rotate_counter_clockwise,
    "DoNothing": do_nothing,
}

In [ ]:
protocol = srl.rl_protocols.ActorCritic(
    particle_type=0, 
    actor=actor, 
    critic=critic, 
    task=rod_finding_task, 
    observable=observable, 
    actions=actions
)
rl_trainer = srl.gyms.Gym(
    [protocol],
    loss,
)
rl_trainer.perform_rl_training(
    system_runner=system_runner,
    n_episodes=n_episodes,
    episode_length=episode_length,
)
rl_trainer.export_models()